# Correspondence Analysis (CA): periods, gender, country, occupation, field


## Documentation
* [Correspondence Analysis](https://en.wikipedia.org/wiki/Correspondence_analysis) (Wikipedia)
* [CA - Correspondence Analysis in R: Essentials](https://www.sthda.com/english/articles/31-principal-component-methods-in-r-practical-guide/113-ca-correspondence-analysis-in-r-essentials/)

* [Analyse factorielle des correspondances](https://fr.wikipedia.org/wiki/Analyse_factorielle_des_correspondances) (Wikipedia)
* [Analyse factorielle des correspondances](https://github.com/Sciences-historiques-numeriques/histoire_numerique_methodes/blob/main/statistiques_descriptives/analyse_factorielle_correspondances_manuels.ipynb)


### Add fanalysis library to pip environment

* Open a terminal and activate your data analysis environment:

    -> my_venvs_activate data_analysis
* Install the missing package in the active environment:

    -> pip install fanalysis
* Deactivate the environment when you have finished:

    -> my_venvs_deactivate


In [ ]:
### Activate libraries that will be used in the notebook

import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm

from fanalysis.ca import CA 

import matplotlib.pyplot as plt
import seaborn as sns



In [ ]:
import warnings
warnings.filterwarnings('ignore')


## Create a dataframe with the data to be analysed

We use in this notebook the data produced in the da3-1 chapter, i.e. a list of persons with birth year, gender, place of birth, world country of birth

In [ ]:
csv_address='da_data/da4-AFC.csv'
df_p = pd.read_csv(csv_address)
df_p.head(3)

In [ ]:
df_p = df_p.drop(['CNTR_ID', 'CNTR_NAME'], axis=1)

In [ ]:
### Inspect the dataframe and 
# notably if there are missing values
df_p.info()

In [ ]:
df_p=df_p.rename(columns={'uriPer': 'person_uri'})

In [ ]:
df_p.head(1)

In [ ]:
pd.set_option('display.max_columns', None)
# Reset to default settings if needed later
# pd.reset_option('display.max_columns')

## AFC : Factor analysis

Cf. documentation at the top of the notebook

### Create bivariate summary and CA functions

To make the code more readable and keep it short, we will define several functions here that we will use below.



In [ ]:
### ct_m : contingency table without totals in margins
def bivariate_stats(observed):

    ### Valeurs produites par la fonction de la librairie 'stats'
    statistic, p, dof, expected = stats.chi2_contingency(observed)

    print('Chi-square :', statistic.round(2), ', dof :',dof)
    print('p-value :', p.round(3))

    
    ## Phi-square coefficient = inertia
    phi_2=statistic/observed.sum().sum()
    print('Inertia (Phi-square): ', phi_2.round(3))

    ### Coéfficient de Cramer
    vc = stats.contingency.association(observed, method='cramer')
    print('Cramer: ', round(vc, 3))

    ## the returned expected contingency table can be stored in a variable and used in a function
    return expected





In [ ]:
## test if expected contingency table follows the rules for validity of Chi-square test

def check_chi_square_test_validity(observed: pd.DataFrame) -> bool:
    """
    Validates a contingency table based on two criteria:
    1. No cell has a value less than 1.
    2. Not more than 20% of cells have values less than or equal to 5.
    
    Parameters:
    contingency_df (pd.DataFrame): The contingency table.
    
    Returns:
    bool: True if both conditions are met, False otherwise.
    """


    # Get expected contingency table
    statistic, p, dof, expected = stats.chi2_contingency(observed)
    
    # Condition 1: No cell has a value less than 1
    # min().min() gets the global minimum of the dataframe
    min_val = expected.min().min()
    condition_1 = min_val >= 1
    
    # Condition 2: Not more than 20% of cells have values <= 5
    total_cells = expected.size
    if total_cells == 0:
        return False # Or True, depending on how you define empty tables
        
    cells_le_5 = (expected <= 5).sum().sum()
    percent_le_5 = cells_le_5 / total_cells
    
    condition_2 = percent_le_5 <= 0.20

    # result : True or False
    result=condition_1 and condition_2
    
    return print(f"Table valid for Chi-square test: {result}")



In [ ]:
### ct_m : contingency tables with totals in margins
def plot_chi2_residuals(observed, figsize=(9,3)):
    
    ### Get signed resitduals using statmodels (sm)

    # 1. Create the Table object directly from your data
    table = sm.stats.Table(observed)
    
    # 2. Get Adjusted Residuals instantly (no manual formula needed)
    adjusted_resids = table.standardized_resids
    adjusted_resids.round(1)
   

    fig, ax = plt.subplots(figsize=figsize)         
    
    # Create heatmap
    sns.heatmap(
        adjusted_resids.round(1), 
        annot=True,            # Use boolean True to annotate with data values
        cmap="coolwarm", 
        linewidths=.5, 
        ax=ax,
        cbar_kws={'label': 'Residuals'}
    )
    # 3. Fix Label Rotation (Safe Method)
    # This rotates existing ticks without risking a count mismatch
    ax.set_xticklabels(ax.get_xticklabels(), rotation=80, ha='right')
    ax.set_yticklabels(ax.get_yticklabels(), rotation=20, va='center')
    ax.set_title("Adjusted Residual", fontsize=12)
    
    # ax.set_title("Heatmap of Adjusted Residuals (via statsmodels)")
    plt.tight_layout()
    plt.show()

In [ ]:
def print_eigenvalue(afc):

    eig = pd.DataFrame(afc.eig_)

    r1 = round(eig.iloc[0], 3)
    r2 = round(eig.iloc[2], 2)
    s=list(range(1,len(r1)+1))
    r1.index=s
    r2.index=s

    # https://www.statology.org/pandas-subplots/
    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12,3))

    ax1 = r1.plot(kind='bar', ax=axes[0], title='Eigenvalue des axes')
    ax2 = r2.plot(kind='bar', ax=axes[1], title="Frequence cumulative de l'eigenvalue ")


    ax1.bar_label(ax1.containers[0])
    ax2.bar_label(ax2.containers[0])


    # Met les valeurs xticks en vertical
    fig.autofmt_xdate(rotation=0)
    plt.show()

In [ ]:
def contributions_colonnes(afc):
    
    # Informations sur les contributions des colonnes
    df = afc.col_topandas()[['col_contrib_dim1',
                            'col_contrib_dim2',
                            'col_contrib_dim3']]

    r1 = df.iloc[:,0]
    r2 = df.iloc[:,1]
    r3 = df.iloc[:,2]

    fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(12,6), )

    r1.sort_values().plot(kind='barh', ax=axes[0,0], title='Dim.1')
    r2.sort_values().plot(kind='barh', ax=axes[0,1], title='Dim.2')
    r3.sort_values().plot(kind='barh', ax=axes[0,2], title='Dim.3')

    ### Rows
    df = afc.row_topandas()[['row_contrib_dim1',
                            'row_contrib_dim2',
                            'row_contrib_dim3']]
    r1 = df.iloc[:,0]
    r2 = df.iloc[:,1]
    r3 = df.iloc[:,2]

    r1.sort_values().plot(kind='barh', ax=axes[1,0], title='Dim.1')
    r2.sort_values().plot(kind='barh', ax=axes[1,1], title='Dim.2')
    r3.sort_values().plot(kind='barh', ax=axes[1,2], title='Dim.3')

    plt.tight_layout()
    plt.show()


In [ ]:
def code_gender(gender: str):
    if gender == 'féminin':
        output='f'
    else:
        output='m'
    return output    

### Analyse countries and activity periods

In [ ]:
### Inspect coded countries
print(df_p.groupby(by='coded_country').size().sort_values(ascending=False))

In [ ]:
### Inspect activity periods
print(df_p.groupby(by='periodsActivity').size().sort_values(ascending=False))

In [ ]:
### Contingency table: 
# count how many individuals exhibit both of these categories for each of the two variables 
per_vs_country=pd.crosstab(df_p.periodsActivity, df_p.coded_country, margins=True)

## display all columns
pd.set_option('display.max_columns', None)

per_vs_country.iloc[2:]

In [ ]:
observed = per_vs_country.iloc[2:-1, :-1 ]
observed

In [ ]:
check_chi_square_test_validity(observed)

In [ ]:
expected=bivariate_stats(observed)

In [ ]:
pp = plot_chi2_residuals(observed, figsize=(12, 4))

### Analyse factorielle des correspondances (AFC)

In [ ]:
observed

In [ ]:
# Initialise the CA object, then fit (=calculate)
afc = CA(row_labels=observed.index,col_labels=observed.columns)
afc.fit(observed.values)

In [ ]:
### Inertia (Phi-square):  0.085
print_eigenvalue(afc)

In [ ]:
contributions_colonnes(afc)

In [ ]:
# Represent dimension 1 and 2
afc.mapping(num_x_axis=1,num_y_axis=2,figsize=(10,8))

In [ ]:
# Represent dimension 2 and 3
afc.mapping(num_x_axis=2,num_y_axis=3,figsize=(12,5))

On the third dimension we can notably observe the situation of Poland. Compare with Italy and France 

In [ ]:
## Use the adjusted residual to test the interpretation
pp = plot_chi2_residuals(observed, figsize=(10, 4))

## Gender and period vs country

In [ ]:
### Apply function and create new column
df_p['per_gender']= df_p.apply(lambda x: x.periodsActivity +'_'+ code_gender(x.gender), axis=1)

In [ ]:
df_p.head(2)

In [ ]:
### Contingency table: 
# count how many individuals exhibit both of these categories for each of the two variables 
perGen_vs_country=pd.crosstab(df_p.per_gender, df_p.coded_country, margins=True)

perGen_vs_country

In [ ]:
observed = perGen_vs_country.iloc[:-1, :-1 ]

In [ ]:
check_chi_square_test_validity(observed)

In [ ]:
observed = perGen_vs_country.iloc[8:-1, :-1 ]

In [ ]:
check_chi_square_test_validity(observed)

In [ ]:
expected=bivariate_stats(observed)

### AFC

In [ ]:
afc = CA(row_labels=observed.index,col_labels=observed.columns)
afc.fit(observed.values)

In [ ]:
# Represent dimension 1 and 2
print_eigenvalue(afc)

In [ ]:
### Inertia (Phi-square - Eigenvalue):  0.108
print_eigenvalue(afc)

In [ ]:
contributions_colonnes(afc)

In [ ]:
# Represent dimension 1 and 2
afc.mapping(num_x_axis=1,num_y_axis=2,figsize=(12,6))

In [ ]:
# Represent dimension 2 and 3
afc.mapping(num_x_axis=2,num_y_axis=3,figsize=(12,5))

In [ ]:
pp = plot_chi2_residuals(observed, figsize=(12, 6))

In [ ]:
### do not forget the number:
# proportionally more women but in mumber more men
observed


## Occupation and field as additional qualitative variables

In [ ]:
csv_address='da_data/da4-persons-features.csv'
# Explicitly tell pandas which strings to treat as NA (exclude 'NA' from the list)
# By default, pandas treats 'NA', 'N/A', 'NaN', etc. as missing values.
# If you want to override this by specifying a custom list that does NOT include 'NA'.
## df_pfeat = pd.read_csv(csv_address, na_values=['', 'N/A', 'NULL', 'None'])
df_pfeat = pd.read_csv(csv_address)
df_pfeat.head()

In [ ]:
### Observe the 
df_pfeat.info()

In [ ]:
df_p = pd.merge(df_p,df_pfeat, on='person_uri', how='left')

In [ ]:
df_p.info()

In [ ]:
df_p.iloc[50:54]

In [ ]:
### missing values are excluded by default
# To keep them add parameter , dropna=False
df_p.groupby(by='occupation_main', dropna=False).size()


## Country vs main occupation

In [ ]:
### Contingency table: 
# count how many individuals exhibit both of these categories for each of the two variables 
occupation_vs_country=pd.crosstab(df_p.occupation_main, df_p.coded_country, margins=True)

occupation_vs_country

In [ ]:
observed = occupation_vs_country.iloc[:-1, :-1 ]

In [ ]:
check_chi_square_test_validity(observed)

In [ ]:
expected=bivariate_stats(observed)

### AFC

In [ ]:
afc = CA(row_labels=observed.index,col_labels=observed.columns)
afc.fit(observed.values)

In [ ]:
# One only dimension represents the whole inertia/variance
print_eigenvalue(afc)

In [ ]:
### Represent dimension 1: this would normally be represented along the horizontal axis.
# However, the Python library does not anticipate this possibility.
# We therefore represent it on the diagonal, 
# and can observe attraction and repulsion among categories.
afc.mapping(num_x_axis=1,num_y_axis=1,figsize=(8,8))

In [ ]:
pp = plot_chi2_residuals(observed, figsize=(12, 4))

## Period vs main occupation

In [ ]:
### Contingency table: 
# count how many individuals exhibit both of these categories for each of the two variables 
occupation_vs_activityPeriod=pd.crosstab(df_p.occupation_main, df_p.periodsActivity, margins=True)
occupation_vs_activityPeriod

In [ ]:
observed = occupation_vs_activityPeriod.iloc[:-1, :-1 ]

In [ ]:
check_chi_square_test_validity(observed)

In [ ]:
expected=bivariate_stats(observed)

### AFC

In [ ]:
afc = CA(row_labels=observed.index,col_labels=observed.columns)
afc.fit(observed.values)

In [ ]:
# Represent dimension 1 and 2
print_eigenvalue(afc)

In [ ]:
### Represent dimension 1: this would normally be represented along the horizontal axis.
# However, the Python library does not anticipate this possibility.
# We therefore represent it on the diagonal, 
# and can observe attraction and repulsion among categories.
afc.mapping(num_x_axis=1,num_y_axis=1,figsize=(8,8))

In [ ]:
pp = plot_chi2_residuals(observed, figsize=(9, 3))

## Period and country vs main occupation

In [ ]:
### Apply function and create new column
df_p['per_occupation']= df_p.apply(lambda x: np.nan if pd.isna(x.occupation_main) \
                              else x.periodsActivity +'_'+ x.occupation_main, axis=1)

In [ ]:
df_p.head(2)

In [ ]:
### Contingency table: 
# count how many individuals exhibit both of these categories for each of the two variables 
per_occ_vs_country=pd.crosstab(df_p.per_occupation, df_p.coded_country, margins=True)
per_occ_vs_country

In [ ]:
observed = per_occ_vs_country.iloc[:-1, :-1 ]

In [ ]:
check_chi_square_test_validity(observed)

In [ ]:
expected=bivariate_stats(observed)

### AFC

In [ ]:
afc = CA(row_labels=observed.index,col_labels=observed.columns)
afc.fit(observed.values)

In [ ]:
### Inertia (Phi-square - Eigenvalue):  0.083
print_eigenvalue(afc)

In [ ]:
contributions_colonnes(afc)

In [ ]:
# Represent dimension 1 and 2
afc.mapping(num_x_axis=1,num_y_axis=2,figsize=(12,6))

In [ ]:
# Represent dimension 2 and 3
afc.mapping(num_x_axis=2,num_y_axis=3,figsize=(12,6))

In [ ]:
pp = plot_chi2_residuals(observed, figsize=(16, 10))

## Gender and main occupation vs period

In [ ]:
### Apply function and create new column
df_p['occup_gen']= df_p.apply(lambda x: np.nan if pd.isna(x.occupation_main) \
                              else x.occupation_main +'_'+ code_gender(x.gender), axis=1)

In [ ]:
df_p.head(2)

In [ ]:
### Contingency table: 
# count how many individuals exhibit both of these categories for each of the two variables 
per_vs_occGen=pd.crosstab(df_p.periodsActivity, df_p.occup_gen, margins=True)
per_vs_occGen

In [ ]:
observed = per_vs_occGen.iloc[:-1, :-1 ]

In [ ]:
check_chi_square_test_validity(observed)

In [ ]:
expected=bivariate_stats(observed)

### AFC

In [ ]:
afc = CA(row_labels=observed.index,col_labels=observed.columns)
afc.fit(observed.values)

In [ ]:
### Inertia (Phi-square - Eigenvalue):  0.083
print_eigenvalue(afc)

In [ ]:
contributions_colonnes(afc)

In [ ]:
# Represent dimension 1 and 2
afc.mapping(num_x_axis=1,num_y_axis=2,figsize=(12,6))

In [ ]:
pp = plot_chi2_residuals(observed, figsize=(5, 5))

#### Comment
Apparently, the gender distribution follows the general evolution of disciplines and genders, with a specific interest of female scientists for physics

## Secondary occupation

In [ ]:
### Inspect secondary occupations
print(df_p.groupby(by='occupation_sec1').size().sort_values(ascending=False).iloc[:50])

In [ ]:
### We define a function that codes and aggregates the values in order to avoid dispersion

def codeSecOccupation(occupation: str):
    if 'teacher' in occupation \
        or 'professor' in occupation :
        output='professor'
    elif 'pedagogue' in occupation \
        or 'academic' in occupation \
        or 'philosopher' in occupation \
        or 'translator' in occupation \
        or 'historian' in occupation:
        output='academic'
    elif 'photographer' in occupation \
        or 'journalist' in occupation \
        or 'writer' in occupation \
        or 'science-communicator' in occupation:
        output='writer_journ.'
    else:
        output=occupation
    return output                   

In [ ]:
df_p['codedSecOccupation']=df_p.apply(lambda x : np.nan if pd.isna(x.occupation_sec1) \
                              else codeSecOccupation(x.occupation_sec1), axis=1)

In [ ]:
### Inspect secondary occupations
dfca = df_p.groupby(by='codedSecOccupation').size().sort_values(ascending=False)

In [ ]:
dfca.iloc[:14]

In [ ]:
lc=dfca.iloc[:14].index.to_list()
lc

## Secondary occupation vs country

In [ ]:
### Contingency table: 
dfc = df_p[df_p.codedSecOccupation.isin(lc)]
occ_vs_country=pd.crosstab(dfc.coded_country, dfc.codedSecOccupation, margins=True)
occ_vs_country

In [ ]:
observed = occ_vs_country.iloc[:-1, :-1 ]

In [ ]:
check_chi_square_test_validity(observed)

In [ ]:
expected=bivariate_stats(observed)

### AFC

In [ ]:
afc = CA(row_labels=observed.index,col_labels=observed.columns)
afc.fit(observed.values)

In [ ]:
### Inertia (Phi-square - Eigenvalue):  0.083
print_eigenvalue(afc)

In [ ]:
contributions_colonnes(afc)

In [ ]:
# Represent dimension 1 and 2
afc.mapping(num_x_axis=1,num_y_axis=2,figsize=(12,10))

In [ ]:
# Represent dimension 2 and 3
afc.mapping(num_x_axis=2,num_y_axis=3,figsize=(12,10))

In [ ]:
pp = plot_chi2_residuals(observed, figsize=(16, 10))

## Secondary occupation vs period

In [ ]:
### Contingency table: 
dfc = df_p[df_p.codedSecOccupation.isin(lc)]
occ_vs_per=pd.crosstab(dfc.periodsActivity, dfc.codedSecOccupation, margins=True)
occ_vs_per

In [ ]:
observed = occ_vs_per.iloc[:-1, :-1 ]

In [ ]:
check_chi_square_test_validity(observed)

In [ ]:
expected=bivariate_stats(observed)

### AFC

In [ ]:
afc = CA(row_labels=observed.index,col_labels=observed.columns)
afc.fit(observed.values)

In [ ]:
### Inertia (Phi-square - Eigenvalue):  0.083
print_eigenvalue(afc)

In [ ]:
contributions_colonnes(afc)

In [ ]:
# Represent dimension 1 and 2
afc.mapping(num_x_axis=1,num_y_axis=2,figsize=(12,10))

In [ ]:
# Represent dimension 2 and 3
afc.mapping(num_x_axis=2,num_y_axis=3,figsize=(12,10))

In [ ]:
pp = plot_chi2_residuals(observed, figsize=(12,8))

## Secondary occupation (more categories) vs period

In [ ]:
dfca.iloc[11:50]

In [ ]:
lc=dfca.iloc[11:50].index.to_list()
lc

In [ ]:
### Contingency table: 
dfc = df_p[df_p.codedSecOccupation.isin(lc)]
occ_vs_per=pd.crosstab(dfc.periodsActivity, dfc.codedSecOccupation, margins=True)
occ_vs_per

In [ ]:
observed = occ_vs_per.iloc[:-1, :-1 ]

In [ ]:
check_chi_square_test_validity(observed)

### AFC

In [ ]:
afc = CA(row_labels=observed.index,col_labels=observed.columns)
afc.fit(observed.values)

In [ ]:
### Inertia (Phi-square - Eigenvalue):  0.083
print_eigenvalue(afc)

In [ ]:
contributions_colonnes(afc)

In [ ]:
# Represent dimension 1 and 2
afc.mapping(num_x_axis=1,num_y_axis=2,figsize=(12,10))

In [ ]:
# Represent dimension 2 and 3
afc.mapping(num_x_axis=2,num_y_axis=3,figsize=(12,10))

In [ ]:
pp = plot_chi2_residuals(observed, figsize=(16,8))